In [2]:
import os
os.environ["HF_HOME"] = "D:/huggingface_cache"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print("Environment set up")

Environment set up


In [3]:
from datasets import load_dataset

print("Starting download... this may take 15-30 mins")
ds = load_dataset("Marqo/fashion200k", split="data")
print("Download complete")

d:\outfit_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting download... this may take 15-30 mins


Download complete


In [7]:

from collections import Counter

# Count categories without touching images
cat1 = Counter(ds["category1"])
cat2 = Counter(ds["category2"])

print("=== category1 ===")
for k, v in cat1.most_common():
    print(f"{k}: {v}")

print("\n=== category2 ===")
for k, v in cat2.most_common(20):
    print(f"{k}: {v}")
print(type(ds))

=== category1 ===
dresses: 59090
tops: 48834
skirts: 34929
pants: 31644
jackets: 27127

=== category2 ===
mid length skirts: 10872
cocktail dresses: 10750
casual and day dresses: 10654
maxi and long dresses: 10404
mini and short dresses: 10181
prom and formal dresses: 9750
short sleeve tops: 9041
knee length skirts: 8750
long sleeved tops: 8698
shirts: 8281
mini skirts: 7971
blouses: 7745
t-shirts: 7701
sleeveless and tank tops: 7368
gowns: 7351
maxi skirts: 7336
leggings: 6735
leather jackets: 6390
casual jackets: 5767
wide-leg and palazzo pants: 5495
<class 'datasets.arrow_dataset.Dataset'>


In [9]:
meta = ds.remove_columns(["image"])
df = meta.to_pandas()

# Extract base ID to deduplicate (remove _0, _1, _2 suffixes)
df["base_ID"] = df["item_ID"].str.rsplit("_", n=1).str[0]

# Keep only one photo per unique item
df = df.drop_duplicates(subset="base_ID").reset_index(drop=True)

print(f"After dedup: {len(df)} unique items")
print(df["category1"].value_counts())

After dedup: 72382 unique items
category1
dresses    20974
tops       18237
pants      11921
skirts     10668
jackets    10582
Name: count, dtype: int64


In [10]:
for cat in ["tops", "pants", "skirts", "jackets", "dresses"]:
    print(f"\n=== {cat} ===")
    print(df[df["category1"] == cat]["category2"].value_counts().to_string())


=== tops ===
category2
long sleeved tops           3378
short sleeve tops           3307
sleeveless and tank tops    3022
blouses                     2888
shirts                      2887
t-shirts                    2755

=== pants ===
category2
leggings                      2453
wide-leg and palazzo pants    2151
cropped pants                 2114
skinny pants                  1602
straight-leg pants            1502
full length pants             1344
harem pants                    388
cargo pants                    367

=== skirts ===
category2
knee length skirts    3216
mid length skirts     3073
mini skirts           2380
maxi skirts           1999

=== jackets ===
category2
casual jackets              2352
leather jackets             2314
waistcoats and gilets       1665
blazers and suit jackets    1508
padded and down jackets     1141
denim jackets                885
fur jackets                  717

=== dresses ===
category2
mini and short dresses     3705
casual and day dresses

In [6]:
# See sample text for each category
for cat in ["tops", "pants", "skirts", "jackets"]:
    print(f"\n=== {cat} sample texts ===")
    samples = df[df["category1"] == cat]["text"].head(5).tolist()
    for i, t in enumerate(samples):
        print(f"{i+1}. {t}")
    print()


=== tops sample texts ===
1. black and white shirt with a plaid pattern. the shirt has a relaxed fit and features a button-down collar. the material appears to be a blend of cotton and polyester.
2. peacock printed blouse with a high round neckline and long sleeves. the blouse has a loose fit and is made of a shiny material. the peacock print is a mix of brown, black, and white colors with a floral design.
3. blue jean jacket. the jean jacket is made of denim material and has a classic western style. it features a front button closure, two front pockets, and a front yoke. the jean jacket is worn by a woman who is wearing a white shirt underneath.
4. pink and orange tie-dyed blouse with a v-neck and long sleeves. the blouse has a loose fit and is made of a lightweight material. the colors used in the tie-dye are pink, orange, and yellow. the blouse is styled in a bohemian fashion and is suitable for casual wear.
5. white shirt with a black and red pattern. the shirt has a collar and lo

In [9]:
import pandas as pd
import random
random.seed(42)


# Define compatibility rules based on category2
CASUAL_TOPS    = ["t-shirts", "short sleeve tops", 
                  "sleeveless and tank tops", "shirts"]
FORMAL_TOPS    = ["blouses", "long sleeved tops"]

CASUAL_BOTTOMS = ["leggings", "skinny pants", "cropped pants",
                  "mini skirts", "knee length skirts"]
FORMAL_BOTTOMS = ["straight-leg pants", "full length pants",
                  "mid length skirts", "maxi skirts"]

CASUAL_JACKETS = ["casual jackets", "denim jackets"]
FORMAL_JACKETS = ["blazers and suit jackets", "waistcoats and gilets"]
WINTER_JACKETS = ["padded and down jackets", "leather jackets", 
                  "fur jackets"]

# Tag each item with a style group
def get_style(row):
    c2 = row["category2"]
    if c2 in CASUAL_TOPS + CASUAL_BOTTOMS + CASUAL_JACKETS:
        return "casual"
    elif c2 in FORMAL_TOPS + FORMAL_BOTTOMS + FORMAL_JACKETS:
        return "formal"
    elif c2 in WINTER_JACKETS:
        return "winter"
    else:
        return "other"

df["style"] = df.apply(get_style, axis=1)
print(df["style"].value_counts())

style
casual    26973
other     23880
formal    17357
winter     4172
Name: count, dtype: int64


In [11]:
# Updated complete style tagging
CASUAL_TOPS    = ["t-shirts", "short sleeve tops", 
                  "sleeveless and tank tops", "shirts"]
FORMAL_TOPS    = ["blouses", "long sleeved tops"]

CASUAL_BOTTOMS = ["leggings", "skinny pants", "cropped pants",
                  "mini skirts", "knee length skirts"]
FORMAL_BOTTOMS = ["straight-leg pants", "full length pants",
                  "mid length skirts", "maxi skirts"]

# Adding missing ones
PARTY_BOTTOMS  = ["wide-leg and palazzo pants", "harem pants"]
CASUAL_JACKETS = ["casual jackets", "denim jackets"]
FORMAL_JACKETS = ["blazers and suit jackets", "waistcoats and gilets"]
WINTER_JACKETS = ["padded and down jackets", "leather jackets",
                  "fur jackets"]

# Dress styles
CASUAL_DRESSES = ["casual and day dresses", "mini and short dresses"]
FORMAL_DRESSES = ["cocktail dresses", "prom and formal dresses", 
                  "gowns"]
OTHER_DRESSES  = ["maxi and long dresses"]

def get_style(row):
    c2 = row["category2"]
    if c2 in CASUAL_TOPS + CASUAL_BOTTOMS + CASUAL_JACKETS + CASUAL_DRESSES:
        return "casual"
    elif c2 in FORMAL_TOPS + FORMAL_BOTTOMS + FORMAL_JACKETS + FORMAL_DRESSES:
        return "formal"
    elif c2 in WINTER_JACKETS:
        return "winter"
    elif c2 in PARTY_BOTTOMS:
        return "party"
    elif c2 in OTHER_DRESSES:
        return "casual"  # maxi dresses → casual
    else:
        return "other"

df["style"] = df.apply(get_style, axis=1)

print("=== updated style distribution ===")
print(df["style"].value_counts())

print("\n=== remaining other ===")
print(df[df["style"] == "other"]["category2"].value_counts())

=== updated style distribution ===
style
casual    37808
formal    27496
winter     4172
party      2539
other       367
Name: count, dtype: int64

=== remaining other ===
category2
cargo pants    367
Name: count, dtype: int64


In [12]:
# cargo pants are casual
df["style"] = df["style"].replace({"other": "casual"})

print("=== final style distribution ===")
print(df["style"].value_counts())

# Save updated dataframe
df.to_csv("D:/fashion200k_subset.csv", index=False)
print("\nSaved updated CSV with style tags")

=== final style distribution ===
style
casual    38175
formal    27496
winter     4172
party      2539
Name: count, dtype: int64

Saved updated CSV with style tags


In [13]:
# Check what keywords appear in text for seasonal signals
import re

def check_keywords(df, keywords):
    pattern = '|'.join(keywords)
    matches = df[df["text"].str.contains(pattern, case=False, na=False)]
    return len(matches)

summer_keywords   = ["summer", "beach", "lightweight", "breezy", 
                     "flowy", "linen", "cotton", "sleeveless"]
winter_keywords   = ["winter", "warm", "wool", "knit", "padded", 
                     "thick", "fleece", "heavy"]
monsoon_keywords  = ["rain", "monsoon", "waterproof", "layered", 
                     "trench"]

print(f"Rows with summer signals: {check_keywords(df, summer_keywords)}")
print(f"Rows with winter signals: {check_keywords(df, winter_keywords)}")
print(f"Rows with monsoon signals: {check_keywords(df, monsoon_keywords)}")

Rows with summer signals: 21729
Rows with winter signals: 4819
Rows with monsoon signals: 910


In [16]:
summer_keywords  = ["summer", "beach", "lightweight", "breezy",
                    "flowy", "linen", "cotton", "sleeveless"]
winter_keywords  = ["winter", "warm", "wool", "knit", "padded",
                    "thick", "fleece", "heavy"]

def get_season(row):
    text = str(row["text"]).lower()
    c2   = row["category2"]

    # Winter — category2 is strongest signal
    if c2 in ["padded and down jackets", "fur jackets"] or \
       any(k in text for k in winter_keywords):
        return "winter"

    # Summer — from text keywords
    elif any(k in text for k in summer_keywords):
        return "summer"

    # Sleeveless tops and mini skirts are inherently summer
    elif c2 in ["sleeveless and tank tops", "mini skirts"]:
        return "summer"

    # Long sleeves and full length are autumn/spring
    elif c2 in ["long sleeved tops", "full length pants", 
                "maxi skirts", "padded and down jackets"]:
        return "autumn_spring"

    else:
        return "autumn_spring"  # default

df["season"] = df.apply(get_season, axis=1)

print("=== season distribution ===")
print(df["season"].value_counts())

# Save
df.to_csv("D:/fashion200k_subset.csv", index=False)
print("\nSaved with season tags")

=== season distribution ===
season
autumn_spring    42434
summer           23752
winter            6196
Name: count, dtype: int64

Saved with season tags


In [17]:
# Combine style + season into one tag
df["compatibility_tag"] = df["style"] + "_" + df["season"]

print("=== compatibility tags ===")
print(df["compatibility_tag"].value_counts())

=== compatibility tags ===
compatibility_tag
casual_autumn_spring    21358
formal_autumn_spring    17166
casual_summer           14721
formal_summer            8232
winter_autumn_spring     2237
formal_winter            2098
casual_winter            2096
winter_winter            1886
party_autumn_spring      1673
party_summer              750
party_winter              116
winter_summer              49
Name: count, dtype: int64


In [18]:
df["compatibility_tag"]= df["compatibility_tag"].replace({"winter_winter":"winter",})

In [19]:
print(df["compatibility_tag"].value_counts())

compatibility_tag
casual_autumn_spring    21358
formal_autumn_spring    17166
casual_summer           14721
formal_summer            8232
winter_autumn_spring     2237
formal_winter            2098
casual_winter            2096
winter                   1886
party_autumn_spring      1673
party_summer              750
party_winter              116
winter_summer              49
Name: count, dtype: int64


In [20]:
df = df[df["compatibility_tag"] != "winter_summer"]

In [21]:
df["compatibility_tag"].value_counts()

compatibility_tag
casual_autumn_spring    21358
formal_autumn_spring    17166
casual_summer           14721
formal_summer            8232
winter_autumn_spring     2237
formal_winter            2098
casual_winter            2096
winter                   1886
party_autumn_spring      1673
party_summer              750
party_winter              116
Name: count, dtype: int64

In [22]:
df = df[~df["compatibility_tag"].isin(["other_summer","other_winter","other_autumn_spring"])]

In [23]:
df["compatibility_tag"].value_counts()

compatibility_tag
casual_autumn_spring    21358
formal_autumn_spring    17166
casual_summer           14721
formal_summer            8232
winter_autumn_spring     2237
formal_winter            2098
casual_winter            2096
winter                   1886
party_autumn_spring      1673
party_summer              750
party_winter              116
Name: count, dtype: int64

In [24]:
df.to_csv("D:/fashion200k_subset.csv", index=False)
print(f"Saved. Total rows: {len(df)}")

Saved. Total rows: 72333


In [25]:
# Check how many tops and bottoms exist per compatibility tag
tops    = df[df["category1"] == "tops"]
bottoms = df[df["category1"].isin(["pants", "skirts"])]

print("=== tops per tag ===")
print(tops["compatibility_tag"].value_counts())

print("\n=== bottoms per tag ===")
print(bottoms["compatibility_tag"].value_counts())

=== tops per tag ===
compatibility_tag
casual_summer           5734
casual_autumn_spring    5510
formal_autumn_spring    3198
formal_summer           2366
casual_winter            727
formal_winter            702
Name: count, dtype: int64

=== bottoms per tag ===
compatibility_tag
casual_autumn_spring    7950
formal_autumn_spring    5068
casual_summer           3664
formal_summer           2370
party_autumn_spring     1673
party_summer             750
casual_winter            518
formal_winter            480
party_winter             116
Name: count, dtype: int64


In [26]:
print(df["compatibility_tag"].unique())
print(f"\nTotal unique tags: {df['compatibility_tag'].nunique()}")

<ArrowStringArray>
['casual_autumn_spring',        'casual_summer',        'casual_winter',
        'formal_summer', 'formal_autumn_spring',        'formal_winter',
               'winter', 'winter_autumn_spring',         'party_summer',
  'party_autumn_spring',         'party_winter']
Length: 11, dtype: str

Total unique tags: 11


In [30]:
df["compatibility_tag"].value_counts()
df["category1"].unique()

<ArrowStringArray>
['dresses', 'jackets', 'pants', 'skirts', 'tops']
Length: 5, dtype: str

In [34]:
import pandas as pd
import random

random.seed(42)

pairs = []
seen = set()

tags = df["compatibility_tag"].unique()

for tag in tags:

    # -------------------
    # Step 1 — filter by tag + category
    # -------------------
    tag_tops = df[
        (df["compatibility_tag"] == tag) &
        (df["category1"] == "tops")
    ]

    tag_bottoms = df[
        (df["compatibility_tag"] == tag) &
        (df["category1"].isin(["skirts", "pants"]))
    ]

    tag_jackets = df[
        (df["compatibility_tag"] == tag) &
        (df["category1"] == "jackets")
    ]

    # -------------------
    # Winter logic
    # -------------------
    is_winter = "winter" in str(tag)

    # -------------------
    # Skip invalid cases
    # -------------------
    if len(tag_tops) == 0 or len(tag_bottoms) == 0:
        print(f"Skipping tag {tag} → tops: {len(tag_tops)}, bottoms: {len(tag_bottoms)}")
        continue

    # -------------------
    # Positive pairs
    # -------------------
    count = 0
    while count < 500:

        top = tag_tops.sample(1).iloc[0]
        bottom = tag_bottoms.sample(1).iloc[0]

        pair = {
            "item_A": top["item_ID"],
            "item_B": bottom["item_ID"],
            "tag_A": top["compatibility_tag"],
            "tag_B": bottom["compatibility_tag"],
            "label": 1
        }

        # 👉 Add jacket for winter
        if is_winter and len(tag_jackets) > 0:
            jacket = tag_jackets.sample(1).iloc[0]
            pair["item_C"] = jacket["item_ID"]

        pair_id = tuple(pair.values())

        if pair_id in seen:
            continue

        seen.add(pair_id)
        pairs.append(pair)
        count += 1

    # -------------------
    # Negative sampling
    # -------------------
    other_bottoms = df[
        (df["category1"].isin(["skirts", "pants"])) &
        (df["compatibility_tag"] != tag)
    ]

    if len(other_bottoms) == 0:
        continue

    count = 0
    while count < 500:

        top = tag_tops.sample(1).iloc[0]
        bottom = other_bottoms.sample(1).iloc[0]

        pair = {
            "item_A": top["item_ID"],
            "item_B": bottom["item_ID"],
            "tag_A": top["compatibility_tag"],
            "tag_B": bottom["compatibility_tag"],
            "label": 0
        }

        # 👉 Add random jacket (can mismatch too)
        if is_winter and len(tag_jackets) > 0:
            jacket = tag_jackets.sample(1).iloc[0]
            pair["item_C"] = jacket["item_ID"]

        pair_id = tuple(pair.values())

        if pair_id in seen:
            continue

        seen.add(pair_id)
        pairs.append(pair)
        count += 1


# Convert to DataFrame
pairs_df = pd.DataFrame(pairs)

# Summary
print(f"\nTotal pairs: {len(pairs_df)}")
print(f"Positive: {len(pairs_df[pairs_df['label'] == 1])}")
print(f"Negative: {len(pairs_df[pairs_df['label'] == 0])}")

Skipping tag winter → tops: 0, bottoms: 0
Skipping tag winter_autumn_spring → tops: 0, bottoms: 0
Skipping tag party_summer → tops: 0, bottoms: 750
Skipping tag party_autumn_spring → tops: 0, bottoms: 1673
Skipping tag party_winter → tops: 0, bottoms: 116

Total pairs: 6000
Positive: 3000
Negative: 3000
